In [7]:
import h5py
import pandas as pd
import os
import numpy as np

h5_file_path = "met_data.h5"


In [11]:
# import h5py
# import pandas as pd
# import numpy as np
# from datetime import datetime
# import os

# # Set your file path here
# h5_file_path = "met_data.h5"  # Change to your actual file path

# def convert_h5_to_csv_complete(h5_path):
#     """
#     Extract and convert ALL data from the meteorological H5 file to CSV format
#     """
#     print(f"Processing file: {h5_path}")
    
#     # Create output directory
#     output_dir = "met_data_csv"
#     os.makedirs(output_dir, exist_ok=True)
    
#     # Open the H5 file
#     with h5py.File(h5_path, 'r') as f:
#         # Get the base coordinates from the root group name
#         root_group = list(f.keys())[0]  # 'lat63_41_lon10_11'
#         print(f"Processing root group: {root_group}")
        
#         # Extract geographic coordinates from group name
#         coord_parts = root_group.replace('lat', '').replace('lon', '').split('_')
#         lat_range = f"{coord_parts[0]}_{coord_parts[1]}"
#         lon_range = f"{coord_parts[2]}_{coord_parts[3]}"
        
#         # Get all variables (like air_pressure_at_sea_level)
#         var_groups = list(f[root_group].keys())
#         print(f"Found {len(var_groups)} variables: {var_groups}")
        
#         # Process each variable
#         for var_name in var_groups:
#             var_path = f"{root_group}/{var_name}"
            
#             # Check if forecast group exists
#             if 'forecast' in f[var_path]:
#                 forecast_path = f"{var_path}/forecast"
                
#                 # Get all forecast timestamps
#                 forecast_times = list(f[forecast_path].keys())
#                 print(f"Processing variable {var_name}: found {len(forecast_times)} forecast times")
                
#                 # Create a DataFrame to store all data for this variable
#                 all_data = []
                
#                 # Process each forecast timestamp
#                 for forecast_time in forecast_times:
#                     try:
#                         # Path to this specific forecast
#                         time_path = f"{forecast_path}/{forecast_time}"
                        
#                         # Get all datasets within this forecast
#                         datasets = list(f[time_path].keys())
                        
#                         # Extract the main components
#                         if 'block0_values' in datasets and 'axis0' in datasets and 'axis1' in datasets:
#                             values = f[f"{time_path}/block0_values"][()]
#                             axis0 = f[f"{time_path}/axis0"][()]
#                             axis1 = f[f"{time_path}/axis1"][()]
                            
#                             # Extract any other datasets present
#                             other_data = {}
#                             for ds_name in datasets:
#                                 if ds_name not in ['block0_values', 'axis0', 'axis1']:
#                                     other_data[ds_name] = f[f"{time_path}/{ds_name}"][()]
                            
#                             # Create records for each data point
#                             for i, timestamp_ns in enumerate(axis1):
#                                 # Convert timestamp to readable format
#                                 timestamp_sec = timestamp_ns / 1_000_000_000
#                                 date_str = datetime.fromtimestamp(timestamp_sec).strftime('%Y-%m-%d %H:%M:%S')
                                
#                                 for j, model_idx in enumerate(axis0):
#                                     # Create base record
#                                     record = {
#                                         'lat_range': lat_range,
#                                         'lon_range': lon_range,
#                                         'variable': var_name,
#                                         'forecast_issued': forecast_time,
#                                         'forecast_time': date_str,
#                                         'timestamp_ns': timestamp_ns,
#                                         'model_idx': int(model_idx),
#                                         'value': values[i, j]
#                                     }
                                    
#                                     # Add any other dataset values
#                                     for ds_name, ds_values in other_data.items():
#                                         if ds_name == 'block0_items' and np.array_equal(ds_values, axis0):
#                                             continue  # Skip if identical to axis0
                                        
#                                         # Add based on dimensionality
#                                         if len(ds_values.shape) == 1:
#                                             if j < len(ds_values):
#                                                 record[ds_name] = ds_values[j]
#                                         elif len(ds_values.shape) == 2:
#                                             if i < ds_values.shape[0] and j < ds_values.shape[1]:
#                                                 record[f"{ds_name}"] = ds_values[i, j]
                                    
#                                     all_data.append(record)
                        
#                         # Progress indicator
#                         if len(forecast_times) > 20 and len(forecast_times) % 20 == 0:
#                             print(f"Progress: {forecast_times.index(forecast_time) + 1}/{len(forecast_times)}")
                            
#                     except Exception as e:
#                         print(f"Error processing {time_path}: {str(e)}")
                
#                 # Convert to DataFrame
#                 if all_data:
#                     df = pd.DataFrame(all_data)
                    
#                     # Save to CSV
#                     csv_filename = f"{var_name}_forecasts.csv"
#                     csv_path = os.path.join(output_dir, csv_filename)
#                     df.to_csv(csv_path, index=False)
#                     print(f"Saved {len(df)} records to {csv_path}")
                    
#                     # Return sample for first variable only
#                     if var_name == var_groups[0]:
#                         sample_df = df.head()
#             else:
#                 print(f"No forecast data found for {var_name}")
    
#     print(f"\nConversion complete. CSV files saved to {output_dir}/")
#     return sample_df if 'sample_df' in locals() else None

# # Execute the conversion
# sample_data = convert_h5_to_csv_complete(h5_file_path)

# # Display sample data if available
# if sample_data is not None:
#     sample_data

In [12]:
def extract_all_parameters_to_single_csv(h5_path):
    """
    Extract all meteorological parameters from the H5 file into a single CSV file
    """
    print(f"Processing file: {h5_path}")
    
    # Open the H5 file
    with h5py.File(h5_path, 'r') as f:
        # Get the root group name (coordinates)
        root_group = list(f.keys())[0]  # 'lat63_41_lon10_11'
        print(f"Processing root group: {root_group}")
        
        # Extract geographic coordinates from group name
        coord_parts = root_group.replace('lat', '').replace('lon', '').split('_')
        lat_range = f"{coord_parts[0]}_{coord_parts[1]}"
        lon_range = f"{coord_parts[2]}_{coord_parts[3]}"
        
        # Get all meteorological parameters
        met_parameters = list(f[root_group].keys())
        print(f"Found {len(met_parameters)} parameters: {met_parameters}")
        
        # Create a list to store all data records
        all_data = []
        
        # Track the number of processed forecast times
        total_forecasts_processed = 0
        
        # Process each meteorological parameter
        for param_name in met_parameters:
            param_path = f"{root_group}/{param_name}"
            print(f"Processing parameter: {param_name}")
            
            # Check if forecast group exists
            if 'forecast' in f[param_path]:
                forecast_path = f"{param_path}/forecast"
                
                # Get all forecast timestamps (issue times)
                forecast_times = list(f[forecast_path].keys())
                print(f"  Found {len(forecast_times)} forecast timestamps")
                
                # Process each forecast timestamp (when the forecast was issued)
                forecast_count = 0
                for forecast_time in forecast_times:
                    try:
                        # Path to this specific forecast
                        time_path = f"{forecast_path}/{forecast_time}"
                        
                        # Extract the values and axes
                        values = f[f"{time_path}/block0_values"][()]  # Shape (61, 4)
                        axis0 = f[f"{time_path}/axis0"][()]          # Shape (4,) - grid points
                        axis1 = f[f"{time_path}/axis1"][()]          # Shape (61,) - timestamps
                        
                        # Convert each data point to a record
                        for i, timestamp_ns in enumerate(axis1):
                            # Convert nanosecond timestamp to readable format
                            timestamp_sec = timestamp_ns / 1_000_000_000  # Convert to seconds
                            forecast_time_str = datetime.fromtimestamp(timestamp_sec).strftime('%Y-%m-%d %H:%M:%S')
                            
                            for j, grid_point in enumerate(axis0):
                                all_data.append({
                                    'latitude': lat_range,
                                    'longitude': lon_range,
                                    'parameter': param_name,
                                    'forecast_issued': forecast_time,
                                    'forecast_timestamp': forecast_time_str,
                                    'timestamp_ns': timestamp_ns,
                                    'grid_point': int(grid_point),
                                    'value': values[i, j]
                                })
                        
                        forecast_count += 1
                        # Progress indicator
                        if forecast_count % 50 == 0:
                            print(f"  Processed {forecast_count}/{len(forecast_times)} forecasts")
                            
                    except Exception as e:
                        print(f"  Error processing {time_path}: {str(e)}")
                
                total_forecasts_processed += forecast_count
                print(f"  Completed parameter {param_name}: processed {forecast_count} forecasts")
            else:
                print(f"  No forecast data found for {param_name}")
        
        print(f"Creating DataFrame with {len(all_data)} records...")
        
        # Convert to DataFrame
        df = pd.DataFrame(all_data)
        
        # Add parameter units based on the documentation
        unit_map = {
            'air_pressure_at_sea_level': 'Pa',
            'air_temperature_2m': 'K',
            'cloud_area_fraction': 'pu',
            'integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time': 'J/m2s',
            'wind_direction_10m': 'deg',
            'wind_speed_10m': 'm/s'
        }
        
        # Add a units column
        df['unit'] = df['parameter'].map(unit_map)
        
        # Optimize DataFrame to save memory
        # Convert object columns to categories where appropriate
        for col in ['latitude', 'longitude', 'parameter', 'forecast_issued', 'unit']:
            df[col] = df[col].astype('category')
        
        # Save to CSV
        csv_filename = "all_met_forecasts.csv"
        print(f"Saving to {csv_filename}...")
        df.to_csv(csv_filename, index=False)
        print(f"Successfully saved {len(df)} records to {csv_filename}")
        print(f"Total forecasts processed: {total_forecasts_processed}")
        
        # Return sample data
        return df.head()

# # Execute the extraction and concatenation
# sample_data = extract_all_parameters_to_single_csv(h5_file_path)

# # Display sample data
# print("\nSample data from the combined CSV:")
# display(sample_data)

Processing file: met_data.h5
Processing root group: lat63_41_lon10_11
Found 6 parameters: ['air_pressure_at_sea_level', 'air_temperature_2m', 'cloud_area_fraction', 'integral_of_surface_downwelling_shortwave_flux_in_air_wrt_time', 'wind_direction_10m', 'wind_speed_10m']
Processing parameter: air_pressure_at_sea_level
  Found 1445 forecast timestamps
  Processed 50/1445 forecasts
  Processed 100/1445 forecasts
  Processed 150/1445 forecasts
  Processed 200/1445 forecasts
  Processed 250/1445 forecasts
  Processed 300/1445 forecasts
  Processed 350/1445 forecasts
  Processed 400/1445 forecasts
  Processed 450/1445 forecasts
  Processed 500/1445 forecasts
  Processed 550/1445 forecasts
  Processed 600/1445 forecasts
  Processed 650/1445 forecasts
  Processed 700/1445 forecasts
  Processed 750/1445 forecasts
  Processed 800/1445 forecasts
  Processed 850/1445 forecasts
  Processed 900/1445 forecasts
  Processed 950/1445 forecasts
  Processed 1000/1445 forecasts
  Processed 1050/1445 foreca

,latitude,longitude,parameter,forecast_issued,forecast_timestamp,timestamp_ns,grid_point,value,unit
0,63_41,10_11,air_pressure_at_sea_level,2020-01-01T00Z,2019-12-31 19:00:00,1577836800000000000,0,101052.033088,Pa
1,63_41,10_11,air_pressure_at_sea_level,2020-01-01T00Z,2019-12-31 19:00:00,1577836800000000000,1,101030.709559,Pa
2,63_41,10_11,air_pressure_at_sea_level,2020-01-01T00Z,2019-12-31 19:00:00,1577836800000000000,2,101052.033088,Pa
3,63_41,10_11,air_pressure_at_sea_level,2020-01-01T00Z,2019-12-31 19:00:00,1577836800000000000,3,101030.709559,Pa
4,63_41,10_11,air_pressure_at_sea_level,2020-01-01T00Z,2019-12-31 20:00:00,1577840400000000000,0,100974.696324,Pa


In [24]:
import os
import pandas as pd

folder_path = "met_data_csv"

# Loop through all CSV files in the folder
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path, nrows=5)
        print(df.head(3))
        print("\n")
        print("\n")
        print("\n")


# First unit - in Pa
# Second unit in m/s
# Third unit in Area

  lat_range lon_range            variable forecast_issued  \
0     63_41     10_11  air_temperature_2m  2020-01-01T00Z   
1     63_41     10_11  air_temperature_2m  2020-01-01T00Z   
2     63_41     10_11  air_temperature_2m  2020-01-01T00Z   

         forecast_time         timestamp_ns  model_idx       value  
0  2019-12-31 19:00:00  1577836800000000000          0  277.248994  
1  2019-12-31 19:00:00  1577836800000000000          1  277.885171  
2  2019-12-31 19:00:00  1577836800000000000          2  277.503465  






  lat_range lon_range        variable forecast_issued        forecast_time  \
0     63_41     10_11  wind_speed_10m  2020-01-01T00Z  2019-12-31 19:00:00   
1     63_41     10_11  wind_speed_10m  2020-01-01T00Z  2019-12-31 19:00:00   
2     63_41     10_11  wind_speed_10m  2020-01-01T00Z  2019-12-31 19:00:00   

          timestamp_ns  model_idx     value  
0  1577836800000000000          0  5.200468  
1  1577836800000000000          1  4.162697  
2  1577836800000000000

In [26]:
file_path = "rye_generation_and_load.csv"

df = pd.read_csv(file_path)
df.head(3)

,index,Consumption,Solar,Wind
0,2020-01-01T13:00:00.0,26.514689,0.0,40.59
1,2020-01-01T14:00:00.0,28.326960,0.0,67.86
2,2020-01-01T15:00:00.0,23.682207,0.0,116.68


In [27]:
print(len(df)) 
# 820 samples, at 1 hour -> 8280 hour samples

8280
